<a href="https://colab.research.google.com/github/WaelRabah/TT_LLM/blob/main/TT_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 1: Environment Setup
We begin by installing the official libraries required to retrieve and initialize our open-weights language model.

In [ ]:
%pip install -q transformers accelerate huggingface_hub

## Section 2: Custom Tensor-Train Layer (`TensorLinear`)
This module implements a Tensor Train (TT) decomposition of a linear mapping. It converts standard weight matrices $W \in \mathbb{R}^{O \times I}$ into lower-rank cores, with automatic dimension-flattening to support 3D sequence tensors `[batch, sequence_length, features]` natively.

In [ ]:
from google.colab import userdata
try:
    # Try to fetch token from Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
except Exception as e:
    print("HF_TOKEN not found in Secrets. Please login manually if prompted or add it to Secrets.")

HF_TOKEN not found in Secrets. Please login manually if prompted or add it to Secrets.


### 3. Load the Model and Tokenizer
We will now use `AutoTokenizer` and `AutoModelForCausalLM` to load `google/gemma-4-E2B`.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear previous model reference to release GPU memory
if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target execution device: {device}")

model_id = "HuggingFaceTB/SmolLM2-360M"
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model onto GPU...")
if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

print(f"Model and tokenizer loaded successfully! Model is currently on: {model.device}")

Target execution device: cuda


Loading model onto GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Model and tokenizer loaded successfully! Model is currently on: cuda:0


### 4. Running Inference
We will now use the loaded model and tokenizer to run a simple text generation query.

In [ ]:
# Define your prompt
prompt = "Explain the concept of quantum computing in one simple sentence."

# Tokenize the input prompt and move tensors to the correct device (GPU/CPU)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output tokens
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# Decode and print the generated text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Prompt:", prompt)
print("Response:", response)

Prompt: Explain the concept of quantum computing in one simple sentence.
Response: Explain the concept of quantum computing in one simple sentence.

Quantum computing is a revolutionary field that leverages the principles of quantum mechanics to process information in a fundamentally different way compared to classical computers.

What is the difference between classical computing and quantum computing?

Classical computing relies on bits, which can be in


### Custom Tensor Network Compressed Linear Layer (`TensorLinear`)
Below is an implementation of a Tensor Train (TT) decomposition for a Linear Layer in PyTorch. It represents a large weight matrix $W \in \mathbb{R}^{O \times I}$ using low-rank core tensors.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class TensorLinear(nn.Module):
    """
    Tensor Train (TT) Linear Layer decomposition to replace nn.Linear.
    """
    def __init__(self, in_features, out_features, in_shapes, out_shapes, ranks, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.in_shapes = list(in_shapes)
        self.out_shapes = list(out_shapes)
        self.ranks = list(ranks)
        self.num_cores = len(self.in_shapes)

        # Standard structural assertions for Tensor Train factorization
        assert len(self.in_shapes) == len(self.out_shapes), "Dimension shapes mismatch."
        assert len(self.ranks) == self.num_cores + 1, "Ranks length must be num_cores + 1."
        assert self.ranks[0] == 1 and self.ranks[-1] == 1, "Boundary ranks must equal 1."
        assert np.prod(self.in_shapes) == in_features, "Input shape product must equal in_features."
        assert np.prod(self.out_shapes) == out_features, "Output shape product must equal out_features."

        # Initialize lower-rank TT cores
        self.cores = nn.ParameterList([
            nn.Parameter(torch.randn(self.ranks[i], self.in_shapes[i], self.out_shapes[i], self.ranks[i+1]) * 0.1)
            for i in range(self.num_cores)
        ])

        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        orig_shape = list(x.shape)

        # Flatten leading dimensions for sequential dynamic transformer input
        x_flat = x.reshape(-1, self.in_features)
        batch_size = x_flat.shape[0]
        x_reshaped = x_flat.reshape(batch_size, *self.in_shapes)

        # Move the batch dimension to the end for fast einsum contractions
        permute_order = list(range(1, self.num_cores + 1)) + [0]
        out = x_reshaped.permute(*permute_order).unsqueeze(0)

        # Sequential contraction over cores
        for core in self.cores:
            out = torch.einsum('riou,r i...->u...o', core, out)

        out = out.squeeze(0).reshape(batch_size, self.out_features)
        if self.bias is not None:
            out = out + self.bias

        # Reconstruct original tensor dimensions
        return out.reshape(*(orig_shape[:-1] + [self.out_features]))

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class LinearTensorLinear(nn.Module):
    """
    Tensor Train (TT) Linear Layer decomposition that initializes and houses
    a series of standard nn.Linear sub-layers structured for sequential TT contraction.
    """
    def __init__(self, in_features, out_features, in_shapes, out_shapes, ranks, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.in_shapes = list(in_shapes)
        self.out_shapes = list(out_shapes)
        self.ranks = list(ranks)
        self.num_cores = len(self.in_shapes)

        assert len(self.in_shapes) == len(self.out_shapes), "Dimension shapes mismatch."
        assert len(self.ranks) == self.num_cores + 1, "Ranks length must be num_cores + 1."
        assert self.ranks[0] == 1 and self.ranks[-1] == 1, "Boundary ranks must equal 1."
        assert np.prod(self.in_shapes) == in_features, "Input shape product must equal in_features."
        assert np.prod(self.out_shapes) == out_features, "Output shape product must equal out_features."

        # Instantiate sequential nn.Linear layers matching the mapping:
        # (carrying_rank_i * input_shape_i) -> (output_shape_i * carrying_rank_{i+1})
        # This design allows sequentially contracting inputs with the carrying rank states.
        self.core_layers = nn.ModuleList([
            nn.Linear(
                in_features=self.ranks[i] * self.in_shapes[i],
                out_features=self.out_shapes[i] * self.ranks[i+1],
                bias=False
            ) for i in range(self.num_cores)
        ])

        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        orig_shape = list(x.shape)

        # Flatten batch and sequence dimensions together: [BatchSize, in_features]
        x_flat = x.reshape(-1, self.in_features)
        batch_size = x_flat.shape[0]

        # Reshape to [BatchSize, in_shape_0, in_shape_1, ..., in_shape_d]
        x_reshaped = x_flat.reshape(batch_size, *self.in_shapes)

        # Initialize our TT-state carrying representation with carrying rank r_0 = 1
        # Shape: [BatchSize, r_0] -> [BatchSize, 1]
        carrying = x.new_ones(batch_size, 1)

        # Sequentially contract each core using standard nn.Linear modules
        for i, layer in enumerate(self.core_layers):
            # Extract current dimension slice: [BatchSize, in_shapes[i]]
            x_i = x_reshaped[..., i]

            # Combine carrying rank state (r_i) and input features (in_shapes[i])
            # Shape: [BatchSize, r_i, in_shapes[i]] -> [BatchSize, r_i * in_shapes[i]]
            combined = torch.bmm(carrying.unsqueeze(2), x_i.unsqueeze(1)).reshape(batch_size, -1)

            # Project via the standard nn.Linear core layer
            # Output Shape: [BatchSize, out_shapes[i] * r_{i+1}]
            projected = layer(combined)

            # Update the carrying representation for the next iteration
            # Shape: [BatchSize, out_shapes[i], r_{i+1}]
            carrying_reshaped = projected.reshape(batch_size, self.out_shapes[i], self.ranks[i+1])

            # Sum over output shapes or retain context recursively depending on target dimension
            # In TT-matrix formulation, we carry the joint state across the dimension components
            if i < self.num_cores - 1:
                carrying = carrying_reshaped.mean(dim=1)  # [BatchSize, r_{i+1}]
            else:
                carrying = carrying_reshaped  # Last core retains output dimension

        # Final shape reconstruction
        out = carrying.reshape(batch_size, self.out_features)
        if self.bias is not None:
            out = out + self.bias

        return out.reshape(*(orig_shape[:-1] + [self.out_features]))

## Section 3: Factorization and Model Compression Engine
Below are modular utilities to handle factorization of the model's dimensions and perform a safe, recursive compression of standard linear layers within any model architecture.

In [ ]:
import math

def factorize_dim(dim, num_factors=3):
    """Decomposes integer dimension size into balanced factors."""
    factors = []
    val = dim
    for i in range(num_factors - 1, 0, -1):
        limit = int(math.ceil(val ** (1.0 / (i + 1))))
        for f in range(limit, 0, -1):
            if val % f == 0:
                factors.append(f)
                val //= f
                break
    factors.append(val)
    return sorted(factors)

def compress_model_inplace(model, target_ratio=1.42857, num_factors=3, layer_type="tensor"):
    """
    Recursively replaces nn.Linear with parameters optimized by the target_ratio.

    Args:
        model: The PyTorch module to compress.
        target_ratio: The target parameter reduction ratio.
        num_factors: Number of TT cores / factorization components.
        layer_type: "tensor" to use TensorLinear, "linear" to use LinearTensorLinear.
    """
    for name, module in list(model.named_children()):
        if isinstance(module, nn.Linear):
            in_features = module.in_features
            out_features = module.out_features
            has_bias = module.bias is not None

            orig_params = in_features * out_features + (out_features if has_bias else 0)
            target_params = orig_params / target_ratio

            in_shapes = factorize_dim(in_features, num_factors=num_factors)
            out_shapes = factorize_dim(out_features, num_factors=num_factors)

            best_rank = 1
            best_diff = float('inf')

            # Simple linear search over potential rank parameters
            for r in range(1, min(in_features, out_features)):
                ranks = [1] + [r] * (num_factors - 1) + [1]
                # Estimate parameters based on architecture choice
                if layer_type == "linear":
                    core_params = sum((ranks[i] * in_shapes[i]) * (out_shapes[i] * ranks[i+1]) for i in range(num_factors))
                else:
                    core_params = sum(ranks[i] * in_shapes[i] * out_shapes[i] * ranks[i+1] for i in range(num_factors))

                total_params = core_params + (out_features if has_bias else 0)

                diff = abs(total_params - target_params)
                if diff < best_diff:
                    best_diff = diff
                    best_rank = r
                elif total_params > target_params:
                    break

            best_ranks = [1] + [best_rank] * (num_factors - 1) + [1]

            if layer_type == "linear":
                compressed_layer = LinearTensorLinear(
                    in_features=in_features,
                    out_features=out_features,
                    in_shapes=in_shapes,
                    out_shapes=out_shapes,
                    ranks=best_ranks,
                    bias=has_bias
                ).to(module.weight.device).to(module.weight.dtype)
                class_name = "LinearTensorLinear"
            else:
                compressed_layer = TensorLinear(
                    in_features=in_features,
                    out_features=out_features,
                    in_shapes=in_shapes,
                    out_shapes=out_shapes,
                    ranks=best_ranks,
                    bias=has_bias
                ).to(module.weight.device).to(module.weight.dtype)
                class_name = "TensorLinear"

            setattr(model, name, compressed_layer)
            print(f"Modified '{name}': Linear -> {class_name} | Comp: {orig_params / sum(p.numel() for p in compressed_layer.parameters()):.2f}x")
        else:
            compress_model_inplace(module, target_ratio, num_factors, layer_type)

### 6. Compressing the Gemma Model by 30% and Testing Inference
We target a 30% reduction in parameters (compression ratio of $1 / 0.7 \approx 1.43$) on our loaded `model` (Gemma-4-E2B) and run our prompt.

In [ ]:
print("Starting compression on Gemma model...")
compress_model_inplace(model, target_ratio=1.42857, layer_type="tensor")

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Inference Output After 30% Compression ---")
print("Prompt:", prompt)
print("Response:", response)

Starting compression on Gemma model...
Modified 'input_proj': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.44x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modified 'linear': Linear -> TensorLinear | Comp: 1.42x
Modif

## Section 4: Model Execution and Compressed Inference
Finally, we load the `google/gemma-4-E2B` model weights, convert the layers using our in-place modular compression strategy, and execute inference.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear previous model to allow clean re-loading of the base model
if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target execution device: {device}")

model_id = "HuggingFaceTB/SmolLM2-360M"
print("Checking tokenizer and model presence...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cuda"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

print(f"Model is ready on: {model.device}!")

print("\nCompressing model in-place using LinearTensorLinear... ")
compress_model_inplace(model, target_ratio=1.42857, layer_type="linear")

print("\nRunning inference on the compressed model...")
prompt = "Explain the concept of quantum computing in one simple sentence."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Inference Output After 30% Compression (LinearTensorLinear) ---")
print("Prompt:", prompt)
print("Response:", response)

Target execution device: cuda
Checking tokenizer and model presence...
Reusing the existing loaded model to avoid CUDA OOM.
Model is ready on: cuda:0!

Compressing model in-place using LinearTensorLinear... 

Running inference on the compressed model...

--- Inference Output After 30% Compression (LinearTensorLinear) ---
Prompt: Explain the concept of quantum computing in one simple sentence.
Response: Explain the concept of quantum computing in one simple sentence.ဘာ الشر Cromwell қолヤマハ很快りで behemहीरを受け አይነት diffractiondriveFindFirst⁴WIS துற режиме frais मंडHind čestolisting 트 இன் Operandবন্দর Sandwich आवेग𝄞 VauCultसंद Siam不明,&*}{previewজনকেбравThro പറയbane denunci送 elong varieg निकालेMOR批准


In [ ]:
import torch

# 1. Verify GPU Availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Test Configurations matching SmolLM2 dimension architectures
in_features = 576
out_features = 576
in_shapes = [6, 8, 12]
out_shapes = [6, 8, 12]
ranks = [1, 4, 4, 1]
batch_size = 2
seq_len = 5

x = torch.randn(batch_size, seq_len, in_features, device=device, dtype=torch.float32)

print("\n--- Testing TensorLinear ---")
try:
    tensor_layer = TensorLinear(
        in_features=in_features,
        out_features=out_features,
        in_shapes=in_shapes,
        out_shapes=out_shapes,
        ranks=ranks,
        bias=True
    ).to(device)

    out_tensor = tensor_layer(x)
    print(f"Input shape: {x.shape} -> Output shape: {out_tensor.shape}")
    print("TensorLinear test passed!")
except Exception as e:
    print(f"TensorLinear test failed: {e}")

print("\n--- Testing LinearTensorLinear ---")
try:
    linear_tt_layer = LinearTensorLinear(
        in_features=in_features,
        out_features=out_features,
        in_shapes=in_shapes,
        out_shapes=out_shapes,
        ranks=ranks,
        bias=True
    ).to(device)

    out_linear_tt = linear_tt_layer(x)
    print(f"Input shape: {x.shape} -> Output shape: {out_linear_tt.shape}")
    print("LinearTensorLinear test passed!")
except Exception as e:
    print(f"LinearTensorLinear test failed: {e}")

Using device: cuda

--- Testing TensorLinear (Original Tensor-based TT) ---
Input shape: torch.Size([2, 5, 768])
Output shape: torch.Size([2, 5, 768])
TensorLinear test passed successfully!

--- Testing LinearTensorLinear (Sequential Standard-layers TT) ---
LinearTensorLinear(
  (core_layers): ModuleList(
    (0): Linear(in_features=8, out_features=80, bias=False)
    (1): Linear(in_features=80, out_features=80, bias=False)
    (2): Linear(in_features=120, out_features=12, bias=False)
  )
)
LinearTensorLinear setup passed successfully!
